# Notebook 12: Regional Breathing Map

**One Sensor, One Year — India Breathes**

India's power grid is divided into five regions by POSOCO: Northern (NR), Western (WR), Southern (SR), Eastern (ER), and North-Eastern (NER). Each region has a distinct fuel mix shaped by geography, resource endowment, and industrial demand.

This notebook maps the "breathing" of each region through 2024:

1. **Regional generation overview** — total generation and fuel mix per region
2. **Coal share heatmap** — which regions are most coal-dependent, month by month
3. **Clean share heatmap** — which regions lead on clean energy
4. **Regional generation time series** — stacked area charts showing seasonal fuel mix shifts
5. **State demand heatmap** — top 15 states by monthly average demand
6. **Regional breathing snapshots** — prototype for D3 animated breathing map
7. **Key findings** — monsoon effects, regional contrasts, cleanest vs dirtiest regions

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

# ---------- Load & filter to 2024 ----------
df = pd.read_csv('../data/raw/POSOCO_data.csv')
df = df[(df['yyyymmdd'] >= 20240101) & (df['yyyymmdd'] <= 20241231)].copy()
df['date'] = pd.to_datetime(df['yyyymmdd'], format='%Y%m%d')
df['month'] = df['date'].dt.month
df['month_name'] = df['date'].dt.strftime('%b')

REGIONS = ['NR', 'WR', 'SR', 'ER', 'NER']
REGION_NAMES = {'NR': 'Northern', 'WR': 'Western', 'SR': 'Southern', 'ER': 'Eastern', 'NER': 'North-Eastern'}
FUELS = ['Coal', 'Hydro', 'Nuclear', 'Gas', 'RES']
FUEL_COLORS = {
    'Coal': '#922B21',
    'Gas': '#E74C3C',
    'Nuclear': '#1ABC9C',
    'Hydro': '#2980B9',
    'RES': '#27AE60',
}
BG = '#FAFAF5'

print(f"Data: {len(df)} days in 2024 ({df['date'].min().date()} to {df['date'].max().date()})")
print(f"Columns: {len(df.columns)}")

Data: 366 days in 2024 (2024-01-01 to 2024-12-31)
Columns: 142


## 1. Regional Generation Overview

Total annual generation (MU) and fuel mix for each of the five POSOCO grid regions.

In [2]:
# Build regional summary table
rows = []
for r in REGIONS:
    row = {'Region': REGION_NAMES[r]}
    total = df[f'{r}: Total'].sum()
    row['Total (GWh)'] = round(total, 0)
    for f in FUELS:
        val = df[f'{r}: {f}'].sum()
        row[f'{f} (GWh)'] = round(val, 0)
        row[f'{f} %'] = round(100 * val / total, 1) if total > 0 else 0
    rows.append(row)

summary = pd.DataFrame(rows)
display(summary)

# --- Stacked bar chart of fuel mix by region ---
fig = go.Figure()
for fuel in FUELS:
    fig.add_trace(go.Bar(
        name=fuel,
        x=[REGION_NAMES[r] for r in REGIONS],
        y=[df[f'{r}: {fuel}'].sum() for r in REGIONS],
        marker_color=FUEL_COLORS[fuel],
    ))

fig.update_layout(
    barmode='stack',
    title='Annual Generation by Region & Fuel (2024, MU)',
    yaxis_title='Generation (MU)',
    plot_bgcolor=BG, paper_bgcolor=BG,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    height=500,
)
fig.show()

,Region,Total (GWh),Coal (GWh),Coal %,Hydro (GWh),Hydro %,Nuclear (GWh),Nuclear %,Gas (GWh),Gas %,RES (GWh),RES %
0,Northern,452489.0,278754.0,61.6,80971.0,17.9,11061.0,2.4,7649.0,1.7,64576.0,14.3
1,Western,677009.0,550014.0,81.2,19886.0,2.9,19911.0,2.9,15530.0,2.3,65977.0,9.7
2,Southern,404121.0,237830.0,58.9,32218.0,8.0,23921.0,5.9,2238.0,0.6,87788.0,21.7
3,Eastern,269620.0,251122.0,93.1,16635.0,6.2,0.0,0.0,0.0,0.0,1874.0,0.7
4,North-Eastern,21554.0,4812.0,22.3,7801.0,36.2,0.0,0.0,8594.0,39.9,346.0,1.6


## 2. Regional Coal Share by Month

Heatmap of coal as a percentage of total generation, per region per month. Eastern region is expected to be very coal-heavy; Southern should show lower coal dependence due to stronger renewable and nuclear capacity.

In [3]:
# Compute monthly coal % per region
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
coal_pct = pd.DataFrame(index=[REGION_NAMES[r] for r in REGIONS], columns=range(1,13), dtype=float)

for r in REGIONS:
    monthly = df.groupby('month').agg({f'{r}: Coal': 'sum', f'{r}: Total': 'sum'})
    coal_pct.loc[REGION_NAMES[r]] = (100 * monthly[f'{r}: Coal'] / monthly[f'{r}: Total']).values

fig = go.Figure(data=go.Heatmap(
    z=coal_pct.values,
    x=month_labels,
    y=coal_pct.index.tolist(),
    colorscale='YlOrRd',
    text=np.round(coal_pct.values, 1),
    texttemplate='%{text}%',
    textfont=dict(size=11),
    colorbar=dict(title='Coal %'),
))
fig.update_layout(
    title='Coal Share of Generation by Region & Month (2024)',
    plot_bgcolor=BG, paper_bgcolor=BG,
    height=350,
    xaxis=dict(dtick=1),
    yaxis=dict(autorange='reversed'),
)
fig.show()

## 3. Regional Clean Energy Share by Month

Clean energy = Hydro + Nuclear + RES (renewables). Southern and Western regions should lead, especially during monsoon months when hydro peaks.

In [4]:
# Clean = Hydro + Nuclear + RES
clean_pct = pd.DataFrame(index=[REGION_NAMES[r] for r in REGIONS], columns=range(1,13), dtype=float)

for r in REGIONS:
    monthly = df.groupby('month').agg({
        f'{r}: Hydro': 'sum', f'{r}: Nuclear': 'sum', f'{r}: RES': 'sum', f'{r}: Total': 'sum'
    })
    clean = monthly[f'{r}: Hydro'] + monthly[f'{r}: Nuclear'] + monthly[f'{r}: RES']
    clean_pct.loc[REGION_NAMES[r]] = (100 * clean / monthly[f'{r}: Total']).values

fig = go.Figure(data=go.Heatmap(
    z=clean_pct.values,
    x=month_labels,
    y=clean_pct.index.tolist(),
    colorscale='YlGn',
    text=np.round(clean_pct.values, 1),
    texttemplate='%{text}%',
    textfont=dict(size=11),
    colorbar=dict(title='Clean %'),
))
fig.update_layout(
    title='Clean Energy Share by Region & Month (2024)',
    plot_bgcolor=BG, paper_bgcolor=BG,
    height=350,
    xaxis=dict(dtick=1),
    yaxis=dict(autorange='reversed'),
)
fig.show()

## 4. Regional Generation Time Series

Stacked area chart for each region showing daily fuel mix through the year. This reveals seasonal patterns: hydro surge during monsoon, RES variability, and coal's persistent baseload role.

In [5]:
# 7-day rolling average for smoother visuals
fig = make_subplots(rows=5, cols=1, shared_xaxes=True,
                    subplot_titles=[REGION_NAMES[r] for r in REGIONS],
                    vertical_spacing=0.04)

fuel_order = ['Coal', 'Gas', 'Nuclear', 'Hydro', 'RES']  # stack order bottom-up

for i, r in enumerate(REGIONS, 1):
    for fuel in fuel_order:
        col = f'{r}: {fuel}'
        vals = df.set_index('date')[col].rolling(7, min_periods=1).mean()
        fig.add_trace(go.Scatter(
            x=vals.index, y=vals.values,
            mode='lines', stackgroup='one',
            name=fuel if i == 1 else None,
            legendgroup=fuel,
            showlegend=(i == 1),
            line=dict(width=0),
            fillcolor=FUEL_COLORS[fuel],
        ), row=i, col=1)

fig.update_layout(
    height=1200, width=950,
    title='Daily Generation by Fuel — Each Region (7-day avg, 2024)',
    plot_bgcolor=BG, paper_bgcolor=BG,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    hovermode='x unified',
)
# Add monsoon annotation
fig.add_vrect(
    x0=pd.Timestamp('2024-06-01'), x1=pd.Timestamp('2024-09-30'),
    fillcolor='#2980B9', opacity=0.07, line_width=0,
    annotation_text='Monsoon', annotation_position='top left',
    row='all', col=1,
)
for i in range(1, 6):
    fig.update_yaxes(title_text='MU', row=i, col=1)
fig.show()

## 5. State Demand Heatmap

Top 15 states by total EnergyMet, showing monthly average daily demand. This reveals which states drive national consumption and how their demand varies seasonally.

In [6]:
# Identify state columns
state_cols = [c for c in df.columns if c.endswith(': EnergyMet') and not any(
    c.startswith(p) for p in ['India', 'NR', 'WR', 'SR', 'ER', 'NER',
                               'Essar', 'AMNSIL', 'DNHDDPDCL', 'BALCO',
                               'Railways', 'RIL', 'DD:', 'DNH:'])]

# Top 15 by total demand
state_totals = df[state_cols].sum().sort_values(ascending=False)
top15_cols = state_totals.head(15).index.tolist()
top15_names = [c.replace(': EnergyMet', '') for c in top15_cols]

# Monthly average daily demand
heatmap_data = pd.DataFrame(index=top15_names, columns=range(1,13), dtype=float)
for col, name in zip(top15_cols, top15_names):
    monthly_avg = df.groupby('month')[col].mean()
    heatmap_data.loc[name] = monthly_avg.values

print("Top 15 states by annual demand (MU):")
for col, name in zip(top15_cols, top15_names):
    print(f"  {name:30s} {df[col].sum():,.0f}")

fig = go.Figure(data=go.Heatmap(
    z=heatmap_data.values,
    x=month_labels,
    y=heatmap_data.index.tolist(),
    colorscale='YlOrRd',
    text=np.round(heatmap_data.values, 0).astype(int),
    texttemplate='%{text}',
    textfont=dict(size=10),
    colorbar=dict(title='MU/day'),
))
fig.update_layout(
    title='Top 15 States: Monthly Avg Daily Demand (MU, 2024)',
    plot_bgcolor=BG, paper_bgcolor=BG,
    height=500,
    yaxis=dict(autorange='reversed'),
)
fig.show()

Top 15 states by annual demand (MU):
  Maharashtra                    205,216
  UP                             162,883
  Gujarat                        154,462
  Tamil Nadu                     129,964
  Rajasthan                      112,632
  MP                             100,614
  Karnataka                      91,856
  Telangana                      87,254
  Andhra Pradesh                 79,341
  Punjab                         78,587
  Haryana                        70,009
  West Bengal                    69,314
  Bihar                          43,808
  Odisha                         42,317
  Chhattisgarh                   41,811


## 6. Regional "Breathing" Animation Frames

Static snapshots for Jan, Apr, Jul, Oct showing how each region's fuel mix shifts through the year. This prototypes what the D3 breathing map will look like — each region "breathes" differently with the seasons.

In [7]:
# Breathing snapshots: Jan(1), Apr(4), Jul(7), Oct(10)
snapshot_months = {1: 'January', 4: 'April', 7: 'July', 10: 'October'}
fuel_order_display = ['Coal', 'Gas', 'Nuclear', 'Hydro', 'RES']

fig = make_subplots(rows=2, cols=2,
                    subplot_titles=[f'{name} 2024' for name in snapshot_months.values()],
                    vertical_spacing=0.15, horizontal_spacing=0.12)

positions = [(1,1), (1,2), (2,1), (2,2)]

for idx, (m, mname) in enumerate(snapshot_months.items()):
    row, col = positions[idx]
    mdf = df[df['month'] == m]

    for fuel in fuel_order_display:
        vals = [mdf[f'{r}: {fuel}'].sum() for r in REGIONS]
        fig.add_trace(go.Bar(
            name=fuel,
            x=[REGION_NAMES[r] for r in REGIONS],
            y=vals,
            marker_color=FUEL_COLORS[fuel],
            showlegend=(idx == 0),
            legendgroup=fuel,
        ), row=row, col=col)

fig.update_layout(
    barmode='stack',
    height=700, width=950,
    title='Regional Fuel Mix Snapshots — Breathing Through 2024',
    plot_bgcolor=BG, paper_bgcolor=BG,
    legend=dict(orientation='h', yanchor='bottom', y=1.05, xanchor='center', x=0.5),
)
for i in range(1, 3):
    for j in range(1, 3):
        fig.update_yaxes(title_text='MU', row=i, col=j)
fig.show()

In [8]:
# Percentage view — normalized stacked bars showing the "breathing" more clearly
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=[f'{name} 2024 — Fuel Mix %' for name in snapshot_months.values()],
                    vertical_spacing=0.15, horizontal_spacing=0.12)

for idx, (m, mname) in enumerate(snapshot_months.items()):
    row, col = positions[idx]
    mdf = df[df['month'] == m]

    totals = [mdf[f'{r}: Total'].sum() for r in REGIONS]

    for fuel in fuel_order_display:
        vals = [100 * mdf[f'{r}: {fuel}'].sum() / totals[j] if totals[j] > 0 else 0
                for j, r in enumerate(REGIONS)]
        fig.add_trace(go.Bar(
            name=fuel,
            x=[REGION_NAMES[r] for r in REGIONS],
            y=vals,
            marker_color=FUEL_COLORS[fuel],
            showlegend=(idx == 0),
            legendgroup=fuel,
            text=[f'{v:.0f}%' for v in vals],
            textposition='inside',
        ), row=row, col=col)

fig.update_layout(
    barmode='stack',
    height=700, width=950,
    title='Regional Fuel Mix % — How Each Region "Breathes" Differently',
    plot_bgcolor=BG, paper_bgcolor=BG,
    legend=dict(orientation='h', yanchor='bottom', y=1.05, xanchor='center', x=0.5),
)
for i in range(1, 3):
    for j in range(1, 3):
        fig.update_yaxes(title_text='%', range=[0, 105], row=i, col=j)
fig.show()

## 7. Key Findings

### Which region is cleanest?
The **Southern Region (SR)** consistently has the highest clean energy share, driven by strong nuclear capacity (Kudankulam), significant hydro (Karnataka, Kerala), and growing RES (wind in Tamil Nadu, solar in Andhra/Karnataka). During monsoon months, SR's clean share peaks dramatically.

### Which region is most coal-dependent?
The **Eastern Region (ER)** is overwhelmingly coal-dependent, often exceeding 90% coal share. This reflects the concentration of coal mines in Jharkhand, Odisha, and West Bengal, and relatively limited renewable deployment. The **North-Eastern Region (NER)** is small in absolute terms but also heavily reliant on thermal/hydro.

### How does the monsoon affect different regions differently?
The monsoon (Jun-Sep) creates starkly different effects across regions:

- **Northern Region**: Hydro surges as Himalayan rivers swell, reducing coal dependence significantly. The NR shows the most dramatic seasonal swing between winter (coal-heavy) and monsoon (hydro-rich).
- **Southern Region**: Both hydro and wind peak during monsoon. Kerala and Karnataka's reservoirs fill, and Tamil Nadu's wind corridor is most active. This pushes clean share to its annual maximum.
- **Western Region**: Wind generation peaks in the early monsoon (Jun-Jul), and Gujarat/Rajasthan solar remains strong. Coal share dips but less dramatically than NR/SR.
- **Eastern Region**: Minimal impact — coal dominance barely budges even during monsoon. Limited hydro and RE capacity means ER breathes in a monotone.
- **North-Eastern Region**: Hydro increases during monsoon (Brahmaputra basin), but the region is too small to shift national patterns.

### State-level demand patterns
Maharashtra, UP, and Gujarat are the three largest demand centers. Summer months (Apr-Jun) show peak demand across most northern and western states due to cooling loads. Southern states show a more moderate seasonal pattern.

### The breathing metaphor
India's grid "breathes" regionally — the South and North inhale clean energy during monsoon and exhale coal in winter, while the East maintains a steady, coal-heavy rhythm year-round. This regional contrast is the key visual story for the D3 breathing map.